In [ ]:
import pymupdf  # PyMuPDF
from PIL import Image
import imagehash
import io
import os

In [ ]:
def optimize_for_ai_analysis(input_path, output_path, hash_threshold=2):
    # Abrimos o documento original
    doc = pymupdf.open(input_path)
    new_doc = pymupdf.open()
    seen_hashes = set()

    print(f"Analisando {len(doc)} páginas...")

    for i in range(len(doc)):
        page = doc[i]

        # 1. Deduplicação Visual (Hash Perceptual)
        # Renderiza em baixa DPI apenas para gerar a digital da página
        pix = page.get_pixmap(matrix=pymupdf.Matrix(0.5, 0.5))
        img_data = pix.tobytes("png")
        v_hash = str(imagehash.phash(Image.open(io.BytesIO(img_data))))

        if v_hash in seen_hashes:
            continue
        seen_hashes.add(v_hash)

        # 2. Transplante de Conteúdo Limpo
        # Criamos uma página nova no documento de destino
        new_page = new_doc.new_page(width=page.rect.width, height=page.rect.height)

        # O show_pdf_page importa apenas o conteúdo visual,
        # deixando para trás as anotações interativas (Screen, Widgets)
        new_page.show_pdf_page(new_page.rect, doc, i)

        # 3. Remoção total de links/anotações remanescentes via XREF
        # Isso garante que nenhum "fantasma" de anotação seja mantido
        new_doc.xref_set_key(new_page.xref, "Annots", "null")

    # 4. Salvamento Moderno (Sem Linearização)
    # garbage=4: remove fluxos não referenciados e duplicações binárias
    # deflate=True: comprime o conteúdo
    new_doc.save(output_path, garbage=4, deflate=True, clean=True)

    new_doc.close()
    doc.close()
    print(f"Otimização concluída: {output_path}")

In [ ]:
def remove_images_from_pdf(input_path, output_path):
    doc = pymupdf.open(input_path)

    print(f"Processando {len(doc)} páginas...")

    for i, page in enumerate(doc):
        # Lista todas as imagens da página
        image_list = page.get_images(full=True)

        if image_list:
            print(f"  Página {i+1}: removendo {len(image_list)} imagem(ns)")

        # Remove cada imagem via xref
        for img in image_list:
            xref = img[0]
            # Substitui o stream da imagem por um objeto vazio
            doc.xref_set_key(xref, "Subtype", "/Image")
            # Zera o conteúdo do stream substituindo por 1 pixel branco transparente
            doc.update_stream(xref, b"")

        # Remove também quaisquer anotações de imagem (XObject do tipo Form com imagens)
        for block in page.get_text("rawdict")["blocks"]:
            if block.get("type") == 1:  # tipo 1 = imagem inline
                # Redesenha a região da imagem com um retângulo branco
                rect = pymupdf.Rect(block["bbox"])
                page.draw_rect(rect, color=(1, 1, 1), fill=(1, 1, 1))

    doc.save(output_path, garbage=4, deflate=True, clean=True)
    doc.close()
    print(f"Concluído: {output_path}")

In [ ]:
path = r"C:\Users\gabri\Documents\POLI_Docs\Sistemas e Sinais\Aulas\aula10_v3.pdf"
novo_path = r"C:\Users\gabri\Documents\POLI_Docs\Sistemas e Sinais\Aulas\aula10.pdf"

In [ ]:
optimize_for_ai_analysis(
    path,
    novo_path,
)

In [ ]:
remove_images_from_pdf(
    path,
    novo_path,
)

In [ ]:
def process_folder(input_folder, output_folder, mode="optimize"):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print(f"Diretório de saída criado: {output_folder}")

    files = [f for f in os.listdir(input_folder) if f.lower().endswith(".pdf")]
    print(f"Encontrados {len(files)} arquivos PDF em {input_folder}")

    for filename in files:
        input_path = os.path.join(input_folder, filename)
        output_path = os.path.join(output_folder, filename)

        print(f"\nProcessando: {filename}")
        try:
            if mode == "optimize":
                optimize_for_ai_analysis(input_path, output_path)
            elif mode == "remove_images":
                remove_images_from_pdf(input_path, output_path)
            else:
                print(f"Modo desconhecido: {mode}")
        except Exception as e:
            print(f"Erro ao processar {filename}: {e}")

    print("\nProcessamento da pasta concluído.")

In [ ]:
path_folder = r"C:\Users\gabri\Documents\POLI_Docs\Lab Circ\P2"
novo_path_folder = r"C:\Users\gabri\Documents\POLI_Docs\Lab Circ\P2_leve"

In [ ]:
process_folder(path_folder, novo_path_folder, mode="optimize")

In [ ]:
process_folder(path_folder, novo_path_folder, mode="remove_images")